# Milestone 4 Unstructured Data Retrieval

## Goal

Build a traceable retrieval workflow for the Gigavolt PDF corpus. Every retrieved passage retains its original PDF filename and page number so it can be verified before use.

**Important:** Retrieval is evidence discovery, not an authorization to act on high-voltage, fire, coolant, or decommissioning procedures. Verify each result in the cited PDF.

## Setup

Run this notebook from the `Milestone_4` folder. It expects the instructor PDFs in `data/raw/Unstructured Data/` and the scripts from this repository.

### Key assumptions

- The corpus was downloaded from the course repository without modification.
- `pypdf` is installed in the active Python environment.
- The notebook is run top to bottom on the EC2 instance.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'Unstructured Data'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

assert RAW_DIR.exists(), f'Missing source PDFs: {RAW_DIR}'
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw PDFs found: {len(list(RAW_DIR.glob("*.pdf")))}')

## 1. Extract, chunk, and index

The next cell runs the same version-controlled scripts used outside the notebook. Keeping pipeline logic in scripts makes it easier to reproduce and audit.

In [ ]:
commands = [
    ['scripts/01_ingestion/01_extract_pdf_text.py'],
    ['scripts/02_chunking/02_chunk_documents.py'],
    ['scripts/03_embedding/03_generate_embeddings.py'],
    ['scripts/04_vector_db/04_ingest_chromadb.py', '--rebuild'],
]
for command in commands:
    completed = subprocess.run([sys.executable, *command], cwd=PROJECT_ROOT, text=True, capture_output=True, check=True)
    print(completed.stdout.strip())

## 2. Verify generated artifacts

The manifest proves the corpus size and identifies any PDF pages where text extraction was empty. The chunk count is the searchable retrieval population.

In [ ]:
manifest = json.loads((PROCESSED_DIR / 'extraction_manifest.json').read_text(encoding='utf-8'))
chunk_count = sum(1 for _ in (PROCESSED_DIR / 'chunks.jsonl').open(encoding='utf-8'))
print(f"Documents: {manifest['document_count']}")
print(f"Pages extracted: {manifest['page_count']}")
print(f"Empty-text pages: {len(manifest['empty_text_pages'])}")
print(f'Chunks created: {chunk_count}')

## 3. Evaluate retrieval

Each query should return a result that names the original source PDF and page. Open that PDF page before presenting the information as an answer.

In [ ]:
queries = [
    'What are the emergency response stages for confirmed thermal runaway?',
    'What is the lower cutoff for LFP systems during decommissioning?',
    'What evidence is required for a cooling or thermal warranty claim?',
]
for query in queries:
    print('\n' + '=' * 90)
    result = subprocess.run(
        [sys.executable, 'scripts/05_vector_search/05_search_chromadb.py', query, '--top-k', '3'],
        cwd=PROJECT_ROOT, text=True, capture_output=True, check=True
    )
    print(result.stdout)

## 4. Benchmark HNSW retrieval

The benchmark warms up the persistent HNSW index, measures only vector-database retrieval latency, and checks approximate results against exact cosine top-k results.


In [ ]:
benchmark = subprocess.run(
    [sys.executable, 'scripts/06_benchmark/06_benchmark_vector_search.py'],
    cwd=PROJECT_ROOT, text=True, capture_output=True, check=True
)
print(benchmark.stdout)
metrics = json.loads((PROJECT_ROOT / 'output' / 'benchmarks' / 'benchmark_metrics.json').read_text())
metrics

## Checks

- Raw source PDFs remain unchanged.
- Every result exposes a filename and page number.
- At least one top result for each test question comes from the expected authoritative SOP.
- The benchmark reports P50, P95, P99, and Recall@k; report the executed values rather than an untested claim.
- The retrieved excerpt is checked against the original PDF before use.